In [19]:
# name: post_process
# date: 27/07/2026
# author: Toby Alexander

# description: 
# Post processing for glider data analysis. This script reads in the results of the non-linear fitting of the glider data, and performs further analysis and visualisation of the results.
# Includes data validation against soundings and ERA5 reanalysis data.

# required modules:
#   - Python 3.6+ (created with 3.10.19)
#   - cartopy
#   - cdsapi (https://cds.climate.copernicus.eu/how-to-api) for downloading ERA5 reanalysis data
#   - matplotlib
#   - numpy
#   - pandas
#   - sklearn (https://pypi.org/project/scikit-learn/) for DBSCAN clustering
#   - xarray (https://pypi.org/project/xarray/)

In [20]:
import cartopy.crs as ccrs 
import cartopy.feature as cfeature
import cdsapi   # for downloading ERA5 reanalysis data
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN  # DBSCAN - Density-Based Spatial Clustering of Applications with Noise
import xarray as xr     # working with netCDF files

# 1. Load in fitting results and radiosonde data

In [21]:
# Read in csv file of non-linear fitting results obtained from glider_analysis.ipynb.

root = ""
filename = "nonlinfit_results_FLRDDE434.csv"

filepath = root + filename

print(f"file id: {filename}")    

df = pd.read_csv(filepath)

file id: nonlinfit_results_FLRDDE434.csv


In [22]:
# Read in radiosonde data, sourced from the University of Wyoming radiosonde archive (http://weather.uwyo.edu/upperair/sounding.html). 
# The data is in CSV format, and contains the following columns:
# [time, longitude, latitude, pressure_hPa, geopotential height_m, temperature_C, dew point temperature_C, ice point temperature_C, relative humidity_%, humidity wrt ice_%, mixing ratio_g/kg, wind direction_degree, wind speed_m/s]

re_albemarle_folder = ""
re_albemarle_files = "2026032200-03238.csv"

# re_albemarle_data = pd.read_csv(f"{re_albemarle_folder}\\{re_albemarle_files}")
# radiosonde sample selected for 22-03-26 at 00:00 UTC at Albemarle, UK
re_albemarle_data = pd.read_csv(f"{re_albemarle_folder}{re_albemarle_files}")
print(re_albemarle_data)

re_albemarle_data = re_albemarle_data.drop(index=0)  # drop first entry - always seems to be incomplete

re_albemarle_ws = re_albemarle_data["wind speed_m/s"].values.astype(float)
re_albemarle_alt = re_albemarle_data["geopotential height_m"].values.astype(float)
re_albemarle_wdir = re_albemarle_data["wind direction_degree"].values.astype(float)
re_albemarle_pres = re_albemarle_data["pressure_hPa"].values.astype(float)

# Select only data below 2km altitude, as this is the range of interest for the glider data.
mask_2km = re_albemarle_alt <= 2000
re_albemarle_ws = re_albemarle_ws[mask_2km]
re_albemarle_alt = re_albemarle_alt[mask_2km]
re_albemarle_wdir = re_albemarle_wdir[mask_2km]
re_albemarle_pres = re_albemarle_pres[mask_2km]

                     time  longitude  latitude  pressure_hPa  \
0     2026-03-21 23:15:07    -1.8784   55.0192         999.3   
1     2026-03-21 23:15:09    -1.8782   55.0191         996.4   
2     2026-03-21 23:15:11    -1.8782   55.0191         995.1   
3     2026-03-21 23:15:13    -1.8782   55.0192         993.8   
4     2026-03-21 23:15:15    -1.8780   55.0192         992.3   
...                   ...        ...       ...           ...   
2664  2026-03-22 00:43:21    -1.5772   54.9541          15.2   
2665  2026-03-22 00:43:23    -1.5773   54.9540          15.2   
2666  2026-03-22 00:43:25    -1.5773   54.9539          15.1   
2667  2026-03-22 00:43:27    -1.5774   54.9538          15.1   
2668  2026-03-22 00:43:29    -1.5776   54.9537          15.1   

      geopotential height_m  temperature_C  dew point temperature_C  \
0                       142            4.9                      3.0   
1                       165            6.5                      3.3   
2                 

In [23]:
# Set lat and long of radiosonde launch site (in this case it was Albemarle, UK)
lat_albemarle = 55.019
lon_albemarle = -1.878

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         8 non-null      str    
 1   timestamp  8 non-null      float64
 2   time       8 non-null      str    
 3   lat0       8 non-null      float64
 4   long0      8 non-null      float64
 5   thermidx   8 non-null      float64
 6   win        8 non-null      str    
 7   omegaest   8 non-null      float64
 8   omegastd   8 non-null      float64
 9   Rxest      8 non-null      float64
 10  phixest    8 non-null      float64
 11  uest       8 non-null      float64
 12  ustd       8 non-null      float64
 13  x0est      8 non-null      float64
 14  Ryest      8 non-null      float64
 15  phiyest    8 non-null      float64
 16  vest       8 non-null      float64
 17  vstd       8 non-null      float64
 18  y0est      8 non-null      float64
 19  magU       8 non-null      float64
 20  stdmagU    8 non-null    

In [29]:
def stats_period_zdot(df):
    """
    Investigate how period and zdot change over each thermal; returns mean values and standard deviations.
    """
    results = []

    for (flight_id, therm_idx), thermal_df in df.groupby(['id', 'thermidx']):
        # separate lists of indices for each different thermal
        indices = thermal_df.index.tolist()
        # obtaining the omega values, grouping by the flight ID and thermal index
        omega = df.loc[indices, 'omegaest'].values
        P = abs(2 * np.pi / omega)
        # obtaining zdot values, grouping by the flight ID and thermal index
        zdot = df.loc[indices, 'zdot'].values

        results.append({
            'id': flight_id,
            'thermidx': therm_idx,
            'nbins': len(indices),
            'P_mean': P.mean(),
            'zdot_mean': zdot.mean(),
            'P_std': P.std(),
            'zdot_std': zdot.std()
        })

    return pd.DataFrame(results)

stats = stats_period_zdot(df)

In [ ]:
stats

# 2. Estimate upper bound of uncertainty in wind estimates from inter-bin variation

In [ ]:
def calculate_upper_bound_errors(df):
    """
    Calculate weighted upper bound errors for discrete adjacent bin pairs
    within each thermal. Assumes an even number of bins per thermal.
    Effectively calculates the disagreement between adjacent bins in the same thermal soaring window, in an attempt to quantify the uncertainty in the wind estimates.
    """
    for col in ['u_upper', 'v_upper', 'magU_upper', 'dbUdeg_upper']:
        df[col] = np.nan

    params = [
        ('uest',  'ustd',    'u_upper'),
        ('vest',  'vstd',    'v_upper'),
    ]

    # Group by flight ID and thermal index, then iterate through each group.
    for (flight_id, therm_idx), thermal_df in df.groupby(['id', 'thermidx']):
        indices = thermal_df.index.tolist()

        for i in range(0, len(indices) - 1, 2):
            idx1 = indices[i]
            idx2 = indices[i + 1]
            row1 = df.loc[idx1]
            row2 = df.loc[idx2]

            for est_col, std_col, out_col in params:
                delta = abs(row1[est_col] - row2[est_col])
                
                sigma_upper = delta / np.sqrt(2)  # Assuming n=2 for two measurements        

                df.at[idx1, out_col] = sigma_upper
                df.at[idx2, out_col] = sigma_upper

    return df

In [ ]:
df_err = calculate_upper_bound_errors(df)

In [ ]:
# propagate errors to magnitude and direction estimates
df['magU_upper'] = np.sqrt(df['uest']**2 * df['u_upper']**2 + df['vest']**2 * df['v_upper']**2) / df['magU']
df['dbUdeg_upper'] = np.sqrt(df['vest']**2 * df['u_upper']**2 + df['uest']**2 * df['v_upper']**2) / df['magU']**2

In [ ]:
# Take the maximum of the standard deviation from fitting and interbin variation to get a combined error estimate for each parameter.
df['u_combined']    = np.maximum(df['ustd'],    df['u_upper'])
df['v_combined']    = np.maximum(df['vstd'],    df['v_upper'])
df['magU_combined'] = np.sqrt(df['uest']**2 * df['u_combined']**2 + df['vest']**2 * df['v_combined']**2) / df['magU']
df['bUdeg_combined']= np.sqrt(df['vest']**2 * df['u_combined']**2 + df['uest']**2 * df['v_combined']**2) / df['magU']**2

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df.u_combined, bins=20, range=[0, 3], color='blue', alpha=0.7, label='u_combined')
# plt.hist(df.magU_upper, bins=20, range=[0, 3], color='orange', alpha=0.7, label='magU_upper')
plt.hist(df.v_combined, bins=20, range=[0, 3], color='red', alpha=0.7, label='v_combined')
plt.xlabel('std (m/s)')
plt.ylabel('Frequency')
plt.title('Distribution of standard deviations of wind estimates')
plt.legend()
# plt.suptitle(f"ICAO24: {fid}")
plt.grid(True)
plt.show()

In [ ]:
# Construct CDF plots for the model standard deviations of the wind estimates and the interbin variations.
fig, axes = plt.subplots(1, 2)

for ax, col, label in zip(axes, ['stdmagU', 'magU_upper'], ['Fit error', 'Upper bound']):
    sorted_vals = np.sort(df[col].dropna())
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    ax.plot(sorted_vals, cdf)
    ax.set_xlabel('std (m/s)')
    ax.set_ylabel('CDF')
    ax.set_title(label)
    ax.grid(True)

In [ ]:
# Clean outliers from the dataframe based on standard deviation thresholds for the wind estimates.
# A standard deviation of 1m/s or greater from the model fitting was characterised by poor fits
# Interbin variation of 2.5m/s or greater was considered unphysical when compared to the wind shear observed by the colocated radiosonde data.
df_clean = df[(df['stdmagU'] < 1) & (df['magU_upper'] < 2.5)]

In [ ]:
df_clean.info()

In [ ]:
def plot_qc_diagnostics(df, df_clean, fit_col='stdmagU', upper_col='magU_upper', 
                         alt_col='zhat', flight_col='id', thermal_col='thermidx'):
    """
    Plot quality control diagnostics for the wind estimates, comparing retained and discarded bins based on the cleaning criteria.
    """
    discarded = df[~df.index.isin(df_clean.index)]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 1. Discarded bins by flight
    ax = axes[0, 0]
    flight_counts = df.groupby(flight_col).size()
    flight_counts_clean = df_clean.groupby(flight_col).size()
    flight_counts_discarded = flight_counts - flight_counts_clean.reindex(flight_counts.index, fill_value=0)
    x = np.arange(len(flight_counts))
    ax.bar(x, flight_counts, label='Retained', color='steelblue')
    ax.bar(x, flight_counts_discarded, label='Discarded', color='tomato')
    ax.set_xticks(x)
    ax.set_xticklabels(flight_counts.index, rotation=45, ha='right')
    ax.set_xlabel('Flight ID')
    ax.set_ylabel('Number of bins')
    ax.set_title('Discarded bins by flight')
    ax.legend()

    # 2. Discarded bins by thermal
    ax = axes[0, 1]
    therm_counts = df.groupby(thermal_col).size()
    therm_counts_clean = df_clean.groupby(thermal_col).size()
    therm_counts_discarded = therm_counts - therm_counts_clean.reindex(therm_counts.index, fill_value=0)
    x = np.arange(len(therm_counts))
    ax.bar(x, therm_counts, label='Retained', color='steelblue')
    ax.bar(x, therm_counts_discarded, label='Discarded', color='tomato')
    ax.set_xlabel('Thermal index')
    ax.set_ylabel('Number of bins')
    ax.set_title('Discarded bins by thermal')
    ax.legend()

    # 3. Fit error vs upper bound (retained vs discarded)
    ax = axes[0, 2]
    ax.scatter(df_clean[fit_col], df_clean[upper_col], 
               alpha=0.6, label='Retained', color='steelblue', s=20)
    ax.scatter(discarded[fit_col], discarded[upper_col], 
               alpha=0.6, label='Discarded', color='tomato', s=20)
    lim = max(df[fit_col].max(), df[upper_col].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='1:1 line')
    ax.set_xlabel('Fit error (m/s)')
    ax.set_ylabel('Upper bound (m/s)')
    ax.set_xlim(0, df[fit_col].max() * 1.05)
    ax.set_ylim(0, df[upper_col].max() * 1.05)
    ax.set_title('Fit error vs upper bound')
    ax.legend()

    # 4. Wind speed profile
    ax = axes[1, 0]
    ax.errorbar(df_clean['magU'], df_clean[alt_col], 
                xerr=df_clean[upper_col], fmt='o', alpha=0.5, 
                color='steelblue', ecolor='lightblue', capsize=2, markersize=3,
                label='Retained')
    ax.scatter(discarded['magU'], discarded[alt_col], 
               color='tomato', alpha=0.5, s=20, label='Discarded')
    ax.set_xlabel('Wind speed (m/s)')
    ax.set_ylabel('Altitude (m)')
    ax.set_title('Wind speed profile')
    ax.legend()

    # 5. Wind direction profile
    ax = axes[1, 1]
    ax.errorbar(df_clean['bUdeg'], df_clean[alt_col],
                xerr=df_clean['dbUdeg_upper'], fmt='o', alpha=0.5,
                color='steelblue', ecolor='lightblue', capsize=2, markersize=3,
                label='Retained')
    ax.scatter(discarded['bUdeg'], discarded[alt_col],
               color='tomato', alpha=0.5, s=20, label='Discarded')
    ax.set_xlabel('Wind direction (degrees)')
    ax.set_ylabel('Altitude (m)')
    ax.set_title('Wind direction profile')
    ax.legend()

    # 6. Fit error vs upper bound ratio
    ax = axes[1, 2]
    ratio = df_clean[upper_col] / df_clean[fit_col]
    ax.hist(ratio, bins=20, color='steelblue', edgecolor='white')
    ax.axvline(1, color='tomato', linestyle='--', label='Upper = fit error')
    ax.set_xlabel('Upper bound / fit error')
    ax.set_ylabel('Frequency')
    ax.set_title('Ratio of upper bound to fit error\n(retained bins)')
    ax.legend()

    fig.tight_layout()
    plt.show()

plot_qc_diagnostics(df, df_clean)

# 3. Visualise distribution of measurements, and cluster measurements by spatial and temporal distance

In [ ]:
def plot_clusters_map(df, lat_col='lat0', lon_col='long0',
                      sonde_lat=None, sonde_lon=None, sonde_time=10.75):
    """
    Plot the spatial distribution of wind estimates on a map, optionally including the location of a radiosonde launch.
    """
    plt.rcParams.update({
        'font.size': 12,
        'axes.titlesize': 12,
        'axes.labelsize': 10,
        'xtick.labelsize': 10,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
    })

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
    
    ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='--')
    ax.add_feature(cfeature.RIVERS, linewidth=0.3)
    
    margin = 0.2
    # expand extent to include sonde location
    all_lons = list(df[lon_col]) + ([sonde_lon] if sonde_lon is not None else [])
    all_lats = list(df[lat_col]) + ([sonde_lat] if sonde_lat is not None else [])

    ax.set_extent([min(all_lons) - margin, max(all_lons) + margin,
                min(all_lats) - margin, max(all_lats) + margin])
    
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False

    t_hours = df['timestamp'] % (3600 * 24) / 3600
    vmin, vmax = t_hours.min(), t_hours.max()

    scatter = ax.scatter(df[lon_col], df[lat_col],
                         c=t_hours, cmap='viridis', s=20, alpha=0.7,
                         vmin=vmin, vmax=vmax,
                         transform=ccrs.PlateCarree(),
                         label='Glider measurements')

    # optionally add radiosonde launch point
    if sonde_lat is not None and sonde_lon is not None:
        ax.scatter(sonde_lon, sonde_lat,
                   c=[sonde_time], cmap='viridis', vmin=vmin, vmax=vmax,
                   s=200, marker='+', linewidths=2,
                   transform=ccrs.PlateCarree(),
                   label='Radiosonde launch', zorder=5)

    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label('Time (hr of day)')
    ax.set_title('Spatial distribution of wind estimates')
    ax.legend(loc='upper left')
    fig.tight_layout()
    plt.show()

# call with radiosonde location
plot_clusters_map(df_clean, lat_col='lat0', lon_col='long0',
                  sonde_lat=51.404, sonde_lon=6.968, sonde_time=10.75)

In [ ]:
def cluster_measurements(df, eps_km=50, min_samples=5, 
                          lat_col='lat', lon_col='lon'):
    """
    Cluster measurements by spatial proximity using DBSCAN.
    eps_km: maximum distance between points in the same cluster (km)
    min_samples: minimum number of points to form a cluster
    """
    lat_col = "lat0"
    lon_col = "long0"
    # convert lat/lon to radians for haversine metric
    coords = np.deg2rad(df[[lat_col, lon_col]].values)
    
    # eps in radians (earth radius ~6371km)
    eps_rad = eps_km / 6371.0
    
    db = DBSCAN(eps=eps_rad, min_samples=min_samples, metric='haversine')
    df['cluster'] = db.fit_predict(coords)
    
    # -1 means noise (not assigned to any cluster)
    n_clusters = len(set(df['cluster'])) - (1 if -1 in df['cluster'].values else 0)
    n_noise = (df['cluster'] == -1).sum()
    print(f"Found {n_clusters} clusters, {n_noise} unassigned points")
    
    return df

def plot_clusters_map(df, lat_col='lat0', lon_col='long0',
                      sonde_lat=None, sonde_lon=None):

    plt.rcParams.update({
        'font.size': 12,
        'axes.titlesize': 12,
        'axes.labelsize': 12,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 10,
    })

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
    
    ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='--')
    ax.add_feature(cfeature.RIVERS, linewidth=0.3)
    
    margin = 0.2
    # expand extent to include sonde location
    all_lons = list(df[lon_col]) + ([sonde_lon] if sonde_lon is not None else [])
    all_lats = list(df[lat_col]) + ([sonde_lat] if sonde_lat is not None else [])

    ax.set_extent([min(all_lons) - margin, max(all_lons) + margin,
                min(all_lats) - margin, max(all_lats) + margin])
    
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False

    # plot clusters
    clusters = sorted(df['cluster'].unique())
    cmap = plt.cm.get_cmap('tab10', len(clusters))
    
    for i, cluster_id in enumerate(clusters):
        mask = df['cluster'] == cluster_id
        label = f'Cluster {cluster_id}' if cluster_id != -1 else 'Unassigned'
        color = 'grey' if cluster_id == -1 else cmap(i)
        ax.scatter(df[mask][lon_col], df[mask][lat_col],
                   color=color, label=label, s=20, alpha=0.7,
                   transform=ccrs.PlateCarree())

    # optionally add radiosonde launch point
    if sonde_lat is not None and sonde_lon is not None:
        ax.scatter(sonde_lon, sonde_lat,
                   color='black', s=200, marker='+', linewidths=2,
                   transform=ccrs.PlateCarree(),
                   label='Radiosonde launch', zorder=5)

    ax.legend(bbox_to_anchor=(1.05, 0.5), loc='center left')
    ax.set_title('Spatial clusters of wind estimates')
    fig.tight_layout()
    plt.show()


# run
df_clean = cluster_measurements(df_clean, eps_km=20, min_samples=5)
# call with radiosonde location
plot_clusters_map(df_clean, lat_col='lat0', lon_col='long0',
                  sonde_lat=51.404, sonde_lon=6.968)

In [ ]:
print(df_clean.groupby('cluster').size())

In [ ]:
def plot_wind_profiles(df, alt_col='zhat', speed_col='magU', dir_col='bUdeg',
                       speed_err_col='magU_combined', dir_err_col='bUdeg_combined',
                       cluster_col='cluster'):
    """
    Plot wind speed and direction profiles for each cluster of measurements, including error bars and comparison with radiosonde data.
    """
    clusters = sorted([c for c in df[cluster_col].unique() if c != -1])
    
    for cluster_id in clusters:
        cluster_df = df[df[cluster_col] == cluster_id].copy()
        
        plt.rcParams.update({
        'font.size': 14,          # default for everything
        'axes.titlesize': 14,     # subplot titles
        'axes.labelsize': 14,     # x and y labels
        'xtick.labelsize': 14,    # tick labels
        'ytick.labelsize': 14,
        'legend.fontsize': 14,
        })

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 8), sharey=True)
        fig.suptitle(f'Wind profile — Cluster {cluster_id} (n={len(cluster_df)})')
        
        # wind speed
        ax1.errorbar(cluster_df[speed_col], cluster_df[alt_col],
                     xerr=cluster_df[speed_err_col],
                     fmt='s', markersize=6, capsize=3, linewidth=2,
                     color='steelblue', ecolor='k',
                     alpha=0.8)
        ax1.plot(re_albemarle_ws, re_albemarle_alt, label='Radiosonde (albemarle)', color='darkorange')
        ax1.set_xlabel('Wind speed (m/s)')
        ax1.set_ylabel('Altitude (m)')
        ax1.set_title('Wind speed')
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim(left=0)

        # wind direction
        ax2.errorbar(cluster_df[dir_col], cluster_df[alt_col],
                     xerr=cluster_df[dir_err_col],
                     fmt='s', markersize=6, capsize=3, linewidth=2,
                     color='darkorange', ecolor='k',
                     alpha=0.8)
        ax2.plot(re_albemarle_wdir, re_albemarle_alt, label='Radiosonde (albemarle)', color='darkorange')
        ax2.set_xlabel('Wind direction (degrees)')
        ax2.set_title('Wind direction')
        ax2.grid(True, alpha=0.3)
        ax2.set_xlim(0, 360)
        ax2.set_xticks([0, 90, 180, 270, 360])
        ax2.set_xticklabels(['N', 'E', 'S', 'W', 'N'])

        fig.tight_layout()
        plt.show()

# plot_wind_profiles(df_clean)

In [ ]:
# Plot temporal distribution of measurements in cluster 1, and scatter of time vs altitude to see temporal coverage at each height.

alt_col = 'zhat'

cluster1 = df_clean[df_clean['cluster'] == 1].copy()
cluster1_time = cluster1['timestamp'] % (3600*24) / 3600
cluster1['timestamp_hr'] = cluster1_time

plt.rcParams.update({
    'font.size': 16,          # default for everything
    'axes.titlesize': 16,     # subplot titles
    'axes.labelsize': 14,     # x and y labels
    'xtick.labelsize': 14,    # tick labels
    'ytick.labelsize': 14,
    'legend.fontsize': 16,
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# time distribution
axes[0].hist(cluster1['timestamp_hr'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Time of day (hr)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Temporal distribution of Cluster 1')
axes[0].grid(True, alpha=0.3)

# scatter of time vs altitude to see temporal coverage at each height
scatter = axes[1].scatter(cluster1['timestamp_hr'], cluster1[alt_col],
                           c=cluster1['magU'], cmap='viridis', s=20, alpha=0.7)
cbar = fig.colorbar(scatter, ax=axes[1])
cbar.set_label('Wind speed (m/s)')
axes[1].set_xlabel('Time of day (hr)')
axes[1].set_ylabel('Altitude (m)')
axes[1].set_title('Time vs altitude coverage')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"Cluster 1: {len(cluster1)} points")
print(f"Time range: {cluster1['timestamp_hr'].min():.1f} - {cluster1['timestamp_hr'].max():.1f} hr")
print(f"Altitude range: {cluster1[alt_col].min():.0f} - {cluster1[alt_col].max():.0f} m")

In [ ]:
def subcluster_by_time(df, cluster_id, breaks, cluster_col='cluster'):
    """
    Split a cluster into time-based sub-clusters using manual time breaks.
    breaks: list of hour values defining the boundaries e.g. [12.5, 15.0]
    """
    mask = df[cluster_col] == cluster_id
    df.loc[mask, 'subcluster'] = pd.cut(
        df.loc[mask, 'timestamp_hr'],
        bins=[-np.inf] + breaks + [np.inf],
        labels=[f'{cluster_id}a', f'{cluster_id}b', f'{cluster_id}c']
    )
    return df
 
df_clean['timestamp_hr'] = df_clean['timestamp'] % (3600*24) / 3600

df_clean = subcluster_by_time(df_clean, cluster_id=1, breaks=[12.5, 15.0])

# check counts
print(df_clean[df_clean['cluster'] == 1].groupby('subcluster').size())

In [ ]:
plot_wind_profiles(df_clean[df_clean['cluster'] == 1],
                   cluster_col='subcluster')

# 4. Quantitative comparison between glider measurements and radiosonde

In [ ]:
df_sonde = pd.DataFrame({'magU': re_albemarle_ws, 'direction': re_albemarle_wdir, 'alt': re_albemarle_alt, 'pres': re_albemarle_pres})

In [ ]:
# Convert wind speed and direction measured by sonde to u and v components for comparison with glider estimates.
theta_rad = np.deg2rad(df_sonde['direction'])  # met convention, degrees
df_sonde['u'] = -df_sonde['magU'] * np.sin(theta_rad)
df_sonde['v'] = -df_sonde['magU'] * np.cos(theta_rad)

In [ ]:
# reconstruct magU and direction from converted u/v and compare to original
magU_check = np.sqrt(df_sonde['u']**2 + df_sonde['v']**2)
dir_check = (270 - np.rad2deg(np.arctan2(df_sonde['v'], df_sonde['u']))) % 360

print(np.allclose(magU_check, df_sonde['magU']))        # should be True
print(np.allclose(dir_check, df_sonde['direction']))    # should be True

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.deg2rad, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def compare_to_radiosonde(df_glider, df_sonde,
                           alt_col='alt', zhat_col='zhat',
                           u_col='uest', v_col='vest',
                           u_err_col='u_combined', v_err_col='v_combined',
                           glider_lat_col='lat0', glider_lon_col='long0',
                           sonde_lat=None, sonde_lon=None,
                           sonde_time=None):
    """
    Compare glider wind estimates to nearest radiosonde measurement by altitude.
    Treats radiosonde as truth. Reports fraction of measurements within 1 and 2 sigma.
    """
    results = []

    for _, row in df_glider.iterrows():
        # find nearest radiosonde measurement by altitude
        alt_diff = np.abs(df_sonde[alt_col] - row[zhat_col])
        nearest = df_sonde.loc[alt_diff.idxmin()]

        # compute normalised residuals
        du = row[u_col] - nearest['u']
        dv = row[v_col] - nearest['v']

        # normalise by glider error (treating sonde as truth)
        zu = du / row[u_err_col]
        zv = dv / row[v_err_col]

        # horizontal distance from glider to sonde launch point (km)
        if sonde_lat is not None and sonde_lon is not None:
            horiz_dist_km = haversine_km(row[glider_lat_col], row[glider_lon_col],
                                          sonde_lat, sonde_lon)
            # total 3D distance in metres (combining horizontal and vertical separation)
            vert_diff_m = abs(row[zhat_col] - nearest[alt_col])
            total_dist_m = np.sqrt((horiz_dist_km * 1000)**2 + vert_diff_m**2)
        else:
            horiz_dist_km = np.nan
            total_dist_m  = np.nan

        # time separation in hours
        if sonde_time is not None:
            time_sep_hr = abs(row['timestamp_hr'] - sonde_time)
        else:
            time_sep_hr = np.nan

        results.append({
            'alt':           row[zhat_col],
            'du':            du,
            'dv':            dv,
            'zu':            zu,
            'zv':            zv,
            'sonde_alt':     nearest[alt_col],
            'alt_diff':      alt_diff.min(),
            'horiz_dist_km': horiz_dist_km,
            'total_dist_m':  total_dist_m,
            'time_sep_hr':   time_sep_hr,
        })

    results_df = pd.DataFrame(results)

    # compute fractions within 1 and 2 sigma
    for comp, z_col in [('u', 'zu'), ('v', 'zv')]:
        within_1s = (np.abs(results_df[z_col]) <= 1).mean()
        within_2s = (np.abs(results_df[z_col]) <= 2).mean()
        within_3s = (np.abs(results_df[z_col]) <= 3).mean()
        within_5s = (np.abs(results_df[z_col]) <= 5).mean()
        print(f"{comp}: {within_1s*100:.1f}% within 1σ, {within_2s*100:.1f}% within 2σ, "
              f"{within_3s*100:.1f}% within 3σ, {within_5s*100:.1f}% within 5σ")

    print(f"\nExpected (Gaussian): 68.3% within 1σ, 95.4% within 2σ")
    print(f"\nMean horizontal separation: {results_df['horiz_dist_km'].mean():.1f} km")
    print(f"Mean total 3D separation:   {results_df['total_dist_m'].mean():.0f} m")
    print(f"Mean time separation:       {results_df['time_sep_hr'].mean():.2f} hr")

    return results_df

# call with sonde location and launch time
results_df = compare_to_radiosonde(df_clean[df_clean['subcluster'] == '1a'], df_sonde,
                                    sonde_lat=51.404, sonde_lon=6.968,
                                    sonde_time=10.75)  # 10:45 in decimal hours

In [ ]:
def plot_residuals_vs_separation(results_df):
    """
    Plot absolute residuals of u and v components against time separation and horizontal distance from radiosonde.
    We are trying to see if the disagreement between glider and radiosonde wind estimates increases with time or distance from the radiosonde launch point.
    """
    fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey='row')
    fig.suptitle('Absolute residuals vs separation from radiosonde')

    for col, (ax1, ax2), label in zip(
        ['du', 'dv'],
        [(axes[0, 0], axes[0, 1]), (axes[1, 0], axes[1, 1])],
        ['|du| (m/s)', '|dv| (m/s)']
    ):
        abs_resid = results_df[col].abs()

        # vs time separation
        ax1.scatter(results_df['time_sep_hr'], abs_resid,
                    alpha=0.6, color='steelblue', s=20)
        ax1.set_xlabel('Time separation (hr)')
        ax1.set_ylabel(label)
        ax1.grid(True, alpha=0.3)

        # add linear trend
        mask = results_df['time_sep_hr'].notna() & abs_resid.notna()
        if mask.sum() > 2:
            z = np.polyfit(results_df.loc[mask, 'time_sep_hr'], abs_resid[mask], 1)
            x = np.linspace(results_df['time_sep_hr'].min(),
                            results_df['time_sep_hr'].max(), 100)
            ax1.plot(x, np.polyval(z, x), 'r--', linewidth=1.5, label=f'Trend: {z[0]:.2f} m/s/hr')
            ax1.legend()

        # vs horizontal distance
        ax2.scatter(results_df['horiz_dist_km'], abs_resid,
                    alpha=0.6, color='darkorange', s=20)
        ax2.set_xlabel('Horizontal separation (km)')
        ax2.grid(True, alpha=0.3)

        # add linear trend
        mask = results_df['horiz_dist_km'].notna() & abs_resid.notna()
        if mask.sum() > 2:
            z = np.polyfit(results_df.loc[mask, 'horiz_dist_km'], abs_resid[mask], 1)
            x = np.linspace(results_df['horiz_dist_km'].min(),
                            results_df['horiz_dist_km'].max(), 100)
            ax2.plot(x, np.polyval(z, x), 'r--', linewidth=1.5, label=f'Trend: {z[0]:.2f} m/s/km')
            ax2.legend()

    fig.tight_layout()
    plt.show()

plot_residuals_vs_separation(results_df)

In [ ]:
def plot_sonde_comparison(results_df):
    fig, axes = plt.subplots(1, 3, figsize=(14, 6))

    # normalised residual distributions
    for ax, z_col, label in zip(axes[:2], ['zu', 'zv'], ['u', 'v']):
        ax.hist(results_df[z_col], bins=20, density=True,
                color='steelblue', edgecolor='white', alpha=0.7)
        # overlay standard normal for reference
        x = np.linspace(-5, 5, 100)
        ax.plot(x, np.exp(-x**2/2) / np.sqrt(2*np.pi),
                'r--', label='Standard normal')
        ax.axvline(-1, color='grey', linestyle='--', alpha=0.5)
        ax.axvline( 1, color='grey', linestyle='--', alpha=0.5, label='±1σ')
        ax.axvline(-2, color='grey', linestyle=':',  alpha=0.5)
        ax.axvline( 2, color='grey', linestyle=':',  alpha=0.5, label='±2σ')
        ax.set_xlabel(f'Normalised {label} residual (z)')
        ax.set_ylabel('Density')
        ax.set_title(f'{label} residuals')
        ax.legend()

    # residuals vs altitude
    axes[2].plot(results_df['du'], results_df['alt'], 'o',
                 color='steelblue', alpha=0.6, markersize=4, label='u')
    axes[2].plot(results_df['dv'], results_df['alt'], 'o',
                 color='darkorange', alpha=0.6, markersize=4, label='v')
    axes[2].axvline(0, color='k', linewidth=0.5)
    axes[2].set_xlabel('Residual (m/s)')
    axes[2].set_ylabel('Altitude (m)')
    axes[2].set_title('Residuals vs altitude')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

# run
# results_df = compare_to_radiosonde(df_clean, df_sonde)
# plot_sonde_comparison(results_df)

In [ ]:
def plot_sonde_comparison_abs(results_df):
    fig, axes = plt.subplots(1, 3, figsize=(14, 6))

    # normalised residual distributions
    for ax, z_col, label in zip(axes[:2], ['du', 'dv'], ['u', 'v']):
        ax.hist(results_df[z_col], bins=20, density=True,
                color='steelblue', edgecolor='white', alpha=0.7)
        # overlay standard normal for reference
        x = np.linspace(-5, 5, 100)
        # ax.plot(x, np.exp(-x**2/2) / np.sqrt(2*np.pi),
        #         'r--', label='Standard normal')
        # ax.axvline(-1, color='grey', linestyle='--', alpha=0.5)
        # ax.axvline( 1, color='grey', linestyle='--', alpha=0.5, label='±1σ')
        # ax.axvline(-2, color='grey', linestyle=':',  alpha=0.5)
        # ax.axvline( 2, color='grey', linestyle=':',  alpha=0.5, label='±2σ')
        ax.set_xlabel(f'{label} residual (m/s)')
        ax.set_ylabel('Density')
        ax.set_title(f'{label} residuals')
        ax.legend()

    # residuals vs altitude
    axes[2].plot(results_df['du'], results_df['alt'], 'o',
                 color='steelblue', alpha=0.6, markersize=4, label='u')
    axes[2].plot(results_df['dv'], results_df['alt'], 'o',
                 color='darkorange', alpha=0.6, markersize=4, label='v')
    axes[2].axvline(0, color='k', linewidth=0.5)
    axes[2].set_xlabel('Residual (m/s)')
    axes[2].set_ylabel('Altitude (m)')
    axes[2].set_title('Residuals vs altitude')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

# run
results_df = compare_to_radiosonde(df_clean[df_clean['subcluster'] == '1a'], df_sonde)
plot_sonde_comparison_abs(results_df)

In [ ]:
results_df.info()

# 5. Quantitative comparison between glider measurements and ERA5

In [ ]:
# Connect to the Copernicus Climate Data Store (CDS) API to retrieve additional data if needed.
import cdsapi
c = cdsapi.Client()
print("Connected successfully")

In [ ]:
# Two example retrievals of ERA5 reanalysis data for wind components at various pressure levels. Uncomment and modify as needed to retrieve data.
# The file will be saved in the current working directory.

# c = cdsapi.Client()
# c.retrieve('reanalysis-era5-pressure-levels', {
#     'variable': ['u_component_of_wind', 'v_component_of_wind'],
#     'pressure_level': ['850', '900', '950'],
#     'year': '2023', 'month': '07', 'day': '02',
#     'time': ['10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00'],
#     'area': [52.05, 5.5, 51.2, 7],  # N/W/S/E bounding box
#     'format': 'netcdf'
# }, 'era5_winds.nc')

# c = cdsapi.Client()
# c.retrieve('reanalysis-era5-pressure-levels', {
#     'variable': ['u_component_of_wind', 'v_component_of_wind'],
#     'pressure_level': ['800', '825', '850', '875', '900', '925', '950', '975'],
#     'year': '2023', 'month': '07', 'day': '02',
#     'time': ['09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00'],
#     'area': [52.1, 5.5, 51.0, 7.3],  # N/W/S/E with 0.2 degree margin
#     'format': 'netcdf'
# }, 'era5_winds_02jul.nc')

In [ ]:
# Use xarray function to open and unpack the ERA5 netCDF file containing u and v wind components at various pressure levels for the specified date and time range.
ds = xr.open_dataset('era5_winds_02jul.nc')     # change to correct filename if needed

In [ ]:
print(ds)
print(ds.coords)
print(ds['u'].dims)

In [ ]:
df_clean.info()

In [ ]:
# find what altitude each pressure level of ERA5 corresponds to in the radiosonde
idx = (df_sonde['pres'] - 975).abs().idxmin()
print(df_sonde.loc[idx, ['pres', 'alt']])

idx = (df_sonde['pres'] - 950).abs().idxmin()
print(df_sonde.loc[idx, ['pres', 'alt']])

idx = (df_sonde['pres'] - 925).abs().idxmin()
print(df_sonde.loc[idx, ['pres', 'alt']])

idx = (df_sonde['pres'] - 900).abs().idxmin()
print(df_sonde.loc[idx, ['pres', 'alt']])

idx = (df_sonde['pres'] - 875).abs().idxmin()
print(df_sonde.loc[idx, ['pres', 'alt']])

idx = (df_sonde['pres'] - 850).abs().idxmin()
print(df_sonde.loc[idx, ['pres', 'alt']])

idx = (df_sonde['pres'] - 825).abs().idxmin()
print(df_sonde.loc[idx, ['pres', 'alt']])

idx = (df_sonde['pres'] - 800).abs().idxmin()
print(df_sonde.loc[idx, ['pres', 'alt']])

In [ ]:
def compare_era5_to_glider(df_glider, ds, alt_levels=None,
                            lat_col='lat0', lon_col='long0',
                            time_col='timestamp', alt_col='zhat',
                            u_col='uest', v_col='vest',
                            u_err_col='u_combined', v_err_col='v_combined',
                            max_alt_diff=100):

    # need to define alt_levels mapping pressure levels to approximate altitudes (m) for comparison.

    if alt_levels is None:
        # alt_levels = {975.1: 399.0, 950.0: 618.0, 925.0: 845.0, 900.0: 1074.0, 875.0: 1309.0, 850.0: 1544.0, 825.0: 1784.0, 804.5: 1992.0}
        alt_levels = {975.0: 301, 950.0: 528, 925.0: 751, 900.0: 975, 875.0: 1215.0, 850.0: 1450, 825.0: 1696, 804.5: 1943.0}

    results = []

    for _, row in df_glider.iterrows():

        # find nearest ERA5 altitude level
        nearest_pressure = min(alt_levels, 
                               key=lambda p: abs(alt_levels[p] - row[alt_col]))
        nearest_alt = alt_levels[nearest_pressure]

        # skip if too far from any ERA5 level
        if abs(nearest_alt - row[alt_col]) > max_alt_diff:
            continue

        # spatially interpolate to glider location
        era5_point = ds.sel(pressure_level=nearest_pressure).interp(
            latitude=row[lat_col],
            longitude=row[lon_col],
            valid_time=np.datetime64(pd.Timestamp(row[time_col], unit='s')),
            method='linear'
        )

        u_era5 = float(era5_point['u'].values)
        v_era5 = float(era5_point['v'].values)

        du = row[u_col] - u_era5
        dv = row[v_col] - v_era5
        zu = du / row[u_err_col]
        zv = dv / row[v_err_col]

        results.append({
            'alt':          row[alt_col],
            'era5_alt':     nearest_alt,
            'alt_diff':     abs(nearest_alt - row[alt_col]),
            'du':           du,
            'dv':           dv,
            'zu':           zu,
            'zv':           zv,
            'u_era5':       u_era5,
            'v_era5':       v_era5,
            'u_glider':     row[u_col],
            'v_glider':     row[v_col],
        })

    print(f"Retained {len(results)}/{len(df_glider)} measurements within {max_alt_diff}m of an ERA5 level")
    return pd.DataFrame(results)

results_era5 = compare_era5_to_glider(df_clean, ds)
results_era5 = results_era5.dropna(subset=['u_era5', 'v_era5'])
print(f"Retained {len(results_era5)} valid comparisons")
plot_sonde_comparison(results_era5)
plot_sonde_comparison_abs(results_era5)

In [ ]:
def plot_era5_residuals(results_df):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('ERA5 residuals vs physical variables')

    plot_params = [
        ('alt',       'Altitude (m)',           axes[0, 0], axes[1, 0]),
        ('u_glider',  'Glider wind speed u (m/s)', axes[0, 1], axes[1, 1]),
        ('v_glider',  'Glider wind speed v (m/s)', axes[0, 2], axes[1, 2]),
    ]

    for x_col, x_label, ax_u, ax_v in plot_params:
        x = results_df[x_col]

        # du
        ax_u.scatter(x, results_df['du'].abs(), alpha=0.6, color='steelblue', s=20)
        ax_u.set_xlabel(x_label)
        ax_u.set_ylabel('|du| (m/s)')
        ax_u.axhline(0, color='k', linewidth=0.5)
        ax_u.grid(True, alpha=0.3)

        # dv
        ax_v.scatter(x, results_df['dv'].abs(), alpha=0.6, color='darkorange', s=20)
        ax_v.set_xlabel(x_label)
        ax_v.set_ylabel('|dv| (m/s)')
        ax_v.axhline(0, color='k', linewidth=0.5)
        ax_v.grid(True, alpha=0.3)

        # add linear trend to both
        for ax, col in [(ax_u, 'du'), (ax_v, 'dv')]:
            abs_resid = results_df[col].abs()
            mask = x.notna() & abs_resid.notna()
            if mask.sum() > 2:
                z = np.polyfit(x[mask], abs_resid[mask], 1)
                x_line = np.linspace(x.min(), x.max(), 100)
                ax.plot(x_line, np.polyval(z, x_line),
                        'r--', linewidth=1.5)

    fig.tight_layout()
    plt.show()


def plot_era5_bias_vs_alt(results_df):
    """Plot signed residuals vs altitude to show systematic bias at each ERA5 level."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 8), sharey=True)
    fig.suptitle('ERA5 signed residuals vs altitude')

    # mark ERA5 pressure level altitudes
    era5_alts = results_df['era5_alt'].unique()

    for ax, col, label, color in [
        (ax1, 'du', 'du (m/s)', 'steelblue'),
        (ax2, 'dv', 'dv (m/s)', 'darkorange')
    ]:
        ax.scatter(results_df[col], results_df['alt'],
                   alpha=0.6, color=color, s=20)
        ax.axvline(0, color='k', linewidth=0.5)

        # mark ERA5 level altitudes
        for era5_alt in sorted(era5_alts):
            ax.axhline(era5_alt, color='grey', linewidth=0.8,
                       linestyle='--', alpha=0.5)
            ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] != 0 else -5,
                    era5_alt + 10, f'{era5_alt:.0f}m',
                    fontsize=8, color='grey')

        # mean residual per ERA5 level
        for era5_alt in sorted(era5_alts):
            mask = results_df['era5_alt'] == era5_alt
            mean_resid = results_df.loc[mask, col].mean()
            ax.plot(mean_resid, era5_alt, 'r*', markersize=12,
                    label='Mean per level' if era5_alt == sorted(era5_alts)[0] else '')

        ax.set_xlabel(label)
        ax.set_ylabel('Altitude (m)')
        ax.grid(True, alpha=0.3)
        ax.legend()

    fig.tight_layout()
    plt.show()


# run both
plot_era5_residuals(results_era5)
plot_era5_bias_vs_alt(results_era5)

In [ ]:
def compare_sonde_era5(df_sonde, ds, alt_levels=None):
    """
    Compare radiosonde wind speed with ERA5 at matching pressure levels.
    """
    if alt_levels is None:
        # alt_levels = {975.0: 399.0, 950.0: 618.0, 925.0: 845.0, 900.0: 1074.0, 875.0: 1309.0, 850.0: 1544.0, 825.0: 1784.0, 800.0: 1992.0}
        alt_levels = {975.0: 301, 950.0: 528, 925.0: 751, 900.0: 975, 875.0: 1215.0, 850.0: 1450, 825.0: 1696, 800: 1943.0}


    # radiosonde launch time - 10:45
    sonde_time = pd.Timestamp('2023-07-02 10:45:00')

    results = []

    for pressure, alt in alt_levels.items():
        # get nearest radiosonde measurement by altitude
        idx = (df_sonde['alt'] - alt).abs().idxmin()
        sonde_row = df_sonde.loc[idx]

        # get ERA5 at this pressure level, interpolated to sonde location
        era5_point = ds.sel(pressure_level=pressure).interp(
            latitude=sonde_row['LAT'] if 'LAT' in df_sonde.columns else 51.404 ,
            longitude=sonde_row['LON'] if 'LON' in df_sonde.columns else 6.968,
            valid_time=np.datetime64(sonde_time),
            method='linear'
        )

        u_era5 = float(era5_point['u'].values)
        v_era5 = float(era5_point['v'].values)
        magU_era5 = np.sqrt(u_era5**2 + v_era5**2)

        # radiosonde wind in m/s
        magU_sonde = sonde_row['magU']

        results.append({
            'pressure':   pressure,
            'alt':        alt,
            'magU_sonde': magU_sonde,
            'magU_era5':  magU_era5,
            'u_era5':     u_era5,
            'v_era5':     v_era5,
            'diff':       magU_sonde - magU_era5,
        })

        print(f"{pressure} hPa (~{alt:.0f}m): "
              f"Sonde={magU_sonde:.2f} m/s, "
              f"ERA5={magU_era5:.2f} m/s, "
              f"Diff={magU_sonde - magU_era5:.2f} m/s")

    return pd.DataFrame(results)

sonde_era5_comparison = compare_sonde_era5(df_sonde, ds)

In [ ]:
df_sonde.info()

In [ ]:
def compare_sonde_era5(df_sonde, ds, alt_levels=None,
                        sonde_lat=51.404, sonde_lon=6.968,
                        sonde_time=pd.Timestamp('2023-07-02 10:45:00')):
    """
    Compare radiosonde wind profile with ERA5 at matching pressure levels.
    Plots ERA5 wind speed and direction overlaid on the radiosonde profile.
    """
    if alt_levels is None:
        alt_levels = {975.0: 301, 950.0: 528, 925.0: 751, 900.0: 975,
                      875.0: 1215.0, 850.0: 1450, 825.0: 1696, 800.0: 1943.0}

    results = []

    for pressure, alt in alt_levels.items():
        idx      = (df_sonde['alt'] - alt).abs().idxmin()
        sonde_row = df_sonde.loc[idx]

        era5_point = ds.sel(pressure_level=pressure, method='nearest').interp(
            latitude=sonde_row['LAT'] if 'LAT' in df_sonde.columns else sonde_lat,
            longitude=sonde_row['LON'] if 'LON' in df_sonde.columns else sonde_lon,
            valid_time=np.datetime64(sonde_time),
            method='linear'
        )

        u_era5    = float(era5_point['u'].values)
        v_era5    = float(era5_point['v'].values)
        magU_era5 = np.sqrt(u_era5**2 + v_era5**2)
        dir_era5  = (270 - np.rad2deg(np.arctan2(v_era5, u_era5))) % 360

        results.append({
            'pressure':   pressure,
            'alt':        alt,
            'magU_sonde': sonde_row['magU'],
            'dir_sonde':  sonde_row['bUdeg'] if 'bUdeg' in df_sonde.columns else np.nan,
            'magU_era5':  magU_era5,
            'dir_era5':   dir_era5,
            'u_era5':     u_era5,
            'v_era5':     v_era5,
            'diff':       sonde_row['magU'] - magU_era5,
        })

    results_df = pd.DataFrame(results)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 8), sharey=True)
    fig.suptitle('Radiosonde vs ERA5 wind profile')

    # wind speed
    ax1.plot(df_sonde['magU'], df_sonde['alt'],
             color='steelblue', linewidth=1.5, label='Radiosonde')
    ax1.scatter(results_df['magU_era5'], results_df['alt'],
                color='darkorange', s=80, marker='D', zorder=5,
                label='ERA5')
    ax1.set_xlabel('Wind speed (m/s)')
    ax1.set_ylabel('Altitude (m)')
    ax1.set_title('Wind speed')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(left=0)

    # wind direction
    if 'direction' in df_sonde.columns:
        ax2.plot(df_sonde['direction'], df_sonde['alt'],
                 color='steelblue', linewidth=1.5, label='Radiosonde')
    ax2.scatter(results_df['dir_era5'], results_df['alt'],
                color='darkorange', s=80, marker='D', zorder=5,
                label='ERA5')
    ax2.set_xlabel('Wind direction (degrees)')
    ax2.set_title('Wind direction')
    ax2.set_xlim(0, 360)
    ax2.set_xticks([0, 90, 180, 270, 360])
    ax2.set_xticklabels(['N', 'E', 'S', 'W', 'N'])
    ax2.legend()
    ax2.grid(True, alpha=0.3)   

    fig.tight_layout()
    plt.show()

    return results_df

sonde_era5_comparison = compare_sonde_era5(df_sonde, ds)

In [ ]:
# check what fraction are within more relaxed thresholds
for threshold in [1, 2, 5, 10]:
    frac_u = (results_era5['zu'].abs() < threshold).mean()
    frac_v = (results_era5['zv'].abs() < threshold).mean()
    print(f"|z| < {threshold}: u={frac_u*100:.1f}%, v={frac_v*100:.1f}%")

# identify the outliers
outliers = results_era5[results_era5['zu'].abs() > 10]
print(f"\n{len(outliers)} outliers with |zu| > 10")
# print(outliers[['alt', 'era5_alt', 'alt_diff', 'du', 'dv', 'u_glider', 'u_era5']])
print(outliers)

In [ ]:
print("ERA5 grid extent:")
print(f"Latitude: {float(ds.latitude.min()):.2f} to {float(ds.latitude.max()):.2f}")
print(f"Longitude: {float(ds.longitude.min()):.2f} to {float(ds.longitude.max()):.2f}")

print("\nGlider data extent:")
print(f"Latitude: {df_clean['lat0'].min():.2f} to {df_clean['lat0'].max():.2f}")
print(f"Longitude: {df_clean['long0'].min():.2f} to {float(df_clean['long0'].max()):.2f}")